# DA3 (+Camera) on the NVIDIA PhysicalAI-AV rig — minimal demo

One notebook, end to end: download frames + LiDAR from the
[NVIDIA PhysicalAI-Autonomous-Vehicles](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles)
dataset, load the camera parameters **properly** (true-FOV pinhole for DA3's conditioning,
exact f-theta rays for everything else), run **Depth Anything 3** conditioned on the
calibrated 7-camera rig, recover metric scale from **classical triangulation over the
rig's calibrated baselines** (no LiDAR in the loop), and judge the result against LiDAR.

Background & results: see the write-up at
[wbjang.github.io](https://wbjang.github.io/blog/posts/av-rig-recon/).

**Before running:**
1. Request access to the gated dataset on its
   [HF page](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles) and
   `huggingface-cli login`.
2. Install dependencies (next cell). A GPU with ~4 GB is enough for DA3-Large at the
   resolution used here.

**License note:** the dataset does not permit redistributing imagery — this notebook
ships with outputs cleared; run it yourself to see the frames.

In [1]:
# Uncomment on first run:
# %pip install physical-ai-av DracoPy opencv-python-headless scipy pillow matplotlib imageio
# %pip install "git+https://github.com/ByteDance-Seed/Depth-Anything-3"
# torch: install per https://pytorch.org for your platform

In [2]:
import os, sys, types
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import minimum_filter
from scipy.spatial import cKDTree

sys.modules.setdefault("pycolmap", types.ModuleType("pycolmap"))  # optional DA3 dep
import physical_ai_av
try:
    from depth_anything_3.api import DepthAnything3
except ModuleNotFoundError:
    # alternative to pip-installing DA3: point DA3_SRC at a repo clone's src/ dir
    sys.path.insert(0, os.environ.get("DA3_SRC", "depth-anything-3/src"))
    from depth_anything_3.api import DepthAnything3

RIG7 = ["cross_left", "front_wide", "cross_right", "front_tele",
        "rear_left_70fov", "rear_tele_30fov", "rear_right_70fov"]
SENSOR7 = {n: f"camera_{n}" if n.endswith("fov") else
           {"cross_left": "camera_cross_left_120fov",
            "front_wide": "camera_front_wide_120fov",
            "cross_right": "camera_cross_right_120fov",
            "front_tele": "camera_front_tele_30fov"}[n] for n in RIG7}
SHORT7 = ["cross_left", "front_wide", "cross_right", "front_tele",
          "rear_left", "rear_tele", "rear_right"]
PAL7 = np.array([[0.122, 0.467, 0.706], [0.173, 0.627, 0.173], [1.0, 0.498, 0.055],
                 [0.839, 0.153, 0.157], [0.580, 0.404, 0.741], [0.549, 0.337, 0.294],
                 [0.090, 0.745, 0.812]])
NATIVE_W, NATIVE_H = 1920, 1080
# beyond the real lens FOV, the f-theta polynomial extrapolates -> mask by incidence
MAX_INC7 = {"cross_left": 62.0, "front_wide": 62.0, "cross_right": 62.0,
            "front_tele": 16.0, "rear_left_70fov": 36.0, "rear_tele_30fov": 16.0,
            "rear_right_70fov": 36.0}

# 14 clips (all have LiDAR + all 7 cameras), t0 = a CoC event timestamp (us)
SCENES = {
    "1b818d7e": ("1b818d7e-7759-4479-978a-e4f7bd65e86e", 4068804),
    "21d8bd88": ("21d8bd88-bb2f-45cf-93d0-7ad1b91dc834", 2566686),
    "2ab96e64": ("2ab96e64-9280-4b50-b2b6-2379701d9401", 11032903),
    "31972f64": ("31972f64-e7c7-47f8-8907-466849f5d907", 2999977),
    "4e40e1f9": ("4e40e1f9-2877-446c-a53d-eb7fcbfc05ad", 9876928),
    "5d8b451d": ("5d8b451d-e522-4a5f-8705-389dfce25009", 6219153),
    "7a990bd0": ("7a990bd0-7f4d-4766-b735-effc8088e1eb", 6533311),
    "892eb6e7": ("892eb6e7-83b3-4452-94cd-a49286465682", 3348535),
    "faaee1d8": ("faaee1d8-8530-4761-8ea5-620e54202326", 9011930),
    "02a6d7ea": ("02a6d7ea-2474-4296-811d-d46354b3381b", 2926428),
    "02ad147d": ("02ad147d-9d99-448e-ae30-42e3cbbf94c5", 1713098),
    "045aef98": ("045aef98-d7db-4310-8752-0f49209d5130", 13713002),
    "054c8dba": ("054c8dba-2f98-4d80-aaae-41bacc76db05", 8998275),
    "07721315": ("07721315-227e-40a7-80dd-5e76ff0c21ec", 2981100),
}
PREFIX = "07721315"                     # the clip used in the write-up
CLIP_ID, T0_US = SCENES[PREFIX]

avdi = physical_ai_av.PhysicalAIAVDatasetInterface()
fp = avdi.feature_presence
need = ["lidar_top_360fov"] + [SENSOR7[n] for n in RIG7]
print("clip", CLIP_ID, "| features present:", bool(fp.loc[CLIP_ID, need].all()))

ModuleNotFoundError: No module named 'depth_anything_3'

## 1. Download images (and video)

Each camera feature is a 20 s video; `decode_images_from_timestamps` returns the frame
at-or-before each requested timestamp plus the actual frame timestamps. First access
streams from HF and caches locally.

In [ ]:
frames = {}
for name in RIG7:
    feat = avdi.get_clip_feature(CLIP_ID,
                                 getattr(avdi.features.CAMERA, SENSOR7[name].upper()),
                                 maybe_stream=True)
    imgs, ts = feat.decode_images_from_timestamps(np.asarray([T0_US], dtype=np.int64))
    frames[name] = imgs[0]
    print(f"{name:18s} frame at t={int(ts[0])} us  {imgs[0].shape}")

fig, axes = plt.subplots(2, 4, figsize=(16, 4.6))
for i, name in enumerate(RIG7):
    ax = axes[i // 4, i % 4]
    ax.imshow(frames[name]); ax.set_title(SHORT7[i], fontsize=9)
axes[1, 3].axis("off")
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# a short VIDEO clip from one camera: decode a timestamp range, save as mp4/gif
feat = avdi.get_clip_feature(CLIP_ID, avdi.features.CAMERA.CAMERA_FRONT_WIDE_120FOV,
                             maybe_stream=True)
want = np.arange(T0_US, T0_US + 2_000_000, 100_000, dtype=np.int64)  # 2 s @ 10 Hz
clip_frames, clip_ts = feat.decode_images_from_timestamps(want)
print(f"decoded {len(clip_frames)} frames spanning "
      f"{(clip_ts[-1]-clip_ts[0])/1e6:.1f} s (full clip: "
      f"{(feat.timestamps[-1]-feat.timestamps[0])/1e6:.1f} s, "
      f"{len(feat.timestamps)} frames)")
try:
    import imageio.v3 as iio
    iio.imwrite("front_wide_2s.mp4", clip_frames, fps=10)
    print("wrote front_wide_2s.mp4")
except Exception as e:
    Image.fromarray(clip_frames[0]).save("f0.png")
    print("mp4 writer unavailable ({}); frames are in `clip_frames`".format(e))

## 2. Camera parameters, loaded properly

The dataset has **no pinhole focal**: each camera is an f-theta model (principal point +
degree-4 polynomials angle↔radius). DA3's camera conditioning accepts only
`fov = 2·atan((W/2)/f)` from a pinhole K — so the K must be built so that its FOV equals
the lens's **true** FOV. The naive fit (focal at the optical axis, `fw_poly_1`) announces
the 121.5° lenses as 92°.

In [ ]:
ext = avdi.get_clip_feature(CLIP_ID, avdi.features.CALIBRATION.SENSOR_EXTRINSICS,
                            maybe_stream=True)
intr = avdi.get_clip_feature(CLIP_ID, avdi.features.CALIBRATION.CAMERA_INTRINSICS,
                             maybe_stream=True)
T_c2r = np.stack([ext.sensor_poses[SENSOR7[n]].as_matrix() for n in RIG7])  # cam->rig
cams = [intr.camera_models[SENSOR7[n]] for n in RIG7]                       # f-theta

def rays_from_pixels(cam, pts):
    r = cam.pixel2ray(np.asarray(pts, dtype=np.float64))
    r = r[0] if isinstance(r, tuple) else r
    return r / np.linalg.norm(r, axis=-1, keepdims=True)

def pinhole_K_true(cam):
    cx, cy = cam.principal_point
    row = np.stack([np.arange(NATIVE_W), np.full(NATIVE_W, cy)], -1).astype(float)
    col = np.stack([np.full(NATIVE_H, cx), np.arange(NATIVE_H)], -1).astype(float)
    th_w = np.arccos(np.clip(rays_from_pixels(cam, row)[:, 2], -1, 1)).max()
    th_h = np.arccos(np.clip(rays_from_pixels(cam, col)[:, 2], -1, 1)).max()
    return np.array([[(NATIVE_W / 2) / np.tan(th_w), 0, cx],
                     [0, (NATIVE_H / 2) / np.tan(th_h), cy], [0, 0, 1]])

Ks = np.stack([pinhole_K_true(c) for c in cams])
for i, n in enumerate(RIG7):
    fov_true = np.degrees(2 * np.arctan((NATIVE_W / 2) / Ks[i, 0, 0]))
    fov_axis = np.degrees(2 * np.arctan((NATIVE_W / 2) / cams[i].th2r.coef[1]))
    print(f"{n:18s} fov_w -> DA3: {fov_true:6.1f}°   (naive axis fit would say "
          f"{fov_axis:5.1f}°)")

## 3. LiDAR (Draco-encoded spins) → rig frame

In [ ]:
import DracoPy
df = avdi.get_clip_feature(CLIP_ID, avdi.features.LIDAR.LIDAR_TOP_360FOV,
                           maybe_stream=True)["pointclouds"]
mid = (df["spin_start_timestamp"] + df["spin_end_timestamp"]) / 2
row = df.loc[(mid - T0_US).abs().idxmin()]
pts = np.asarray(DracoPy.decode(bytes(row["draco_encoded_pointcloud"])).points, float)
T_l2r = ext.sensor_poses["lidar_top_360fov"].as_matrix()
pts_rig = pts @ T_l2r[:3, :3].T + T_l2r[:3, 3]
print(f"spin: {len(pts_rig)} points, mid-spin offset from t0: "
      f"{(float(mid.loc[row.name]) - T0_US)/1e3:.0f} ms")

## 4. DA3 (+Camera): conditioned on the calibrated rig

`align_to_input_ext_scale=True` returns depth scaled by matching DA3's predicted camera
translations to the metric input ones — a poor meter (a ~2 m rig for a 5–80 m scene),
replaced in the next section. NOTE: the bare `DepthAnything3()` constructor is
random-init; weights only come via `from_pretrained`.

In [ ]:
paths = []
for name in RIG7:                      # DA3's API takes image paths
    p = f"frame_{name}.png"
    Image.fromarray(frames[name]).save(p)
    paths.append(p)

model = DepthAnything3.from_pretrained("depth-anything/DA3-LARGE-1.1").to("cuda")
w2c = np.stack([np.linalg.inv(T) for T in T_c2r]).astype(np.float32)
with torch.no_grad():
    pred = model.inference(image=paths, extrinsics=w2c,
                           intrinsics=Ks.astype(np.float32),
                           align_to_input_ext_scale=True, process_res=504)
depth = np.asarray(pred.depth, dtype=np.float32)
conf = np.asarray(pred.conf, dtype=np.float32)
Hp, Wp = depth.shape[1:]
print("depth grids:", depth.shape)

## 5. The meter: classical triangulation over the calibrated baselines

Two-view geometry from images is scale-ambiguous; the meter enters through the rig's
calibrated inter-camera distances (1.3–2.5 m). SIFT matches → exact f-theta rays →
midpoint triangulation → per-camera scale `s = median(DA3 depth / triangulated z)`
(divide the depth by it). Tele cameras (0.1 m baselines) get a depth transfer from their
corrected neighbor; anchors past 50 m are gated out (DA3's far-field depth is the weak
part, not the anchors).

In [ ]:
FRONT_PAIRS = [("front_wide", "cross_left"), ("front_wide", "cross_right")]
REAR_PAIRS = [("rear_left_70fov", "rear_tele_30fov"),
              ("rear_tele_30fov", "rear_right_70fov"),
              ("rear_left_70fov", "rear_right_70fov")]
GATE_M, MIN_PTS = 50.0, 10
sift = cv2.SIFT_create(nfeatures=8000)
bf = cv2.BFMatcher()
gray = {n: cv2.cvtColor(frames[n], cv2.COLOR_RGB2GRAY) for n in RIG7}
feats = {n: sift.detectAndCompute(gray[n], None) for n in RIG7}

def match(a, b, ratio=0.75):
    ka, da = feats[a]; kb, db = feats[b]
    good = [m for m, n2 in bf.knnMatch(da, db, k=2) if m.distance < ratio * n2.distance]
    return (np.float64([ka[m.queryIdx].pt for m in good]),
            np.float64([kb[m.trainIdx].pt for m in good]))

def depth_at(dep, pix):
    u = (pix[:, 0] * Wp / NATIVE_W).astype(int).clip(0, Wp - 1)
    v = (pix[:, 1] * Hp / NATIVE_H).astype(int).clip(0, Hp - 1)
    return dep[v, u]

ratios = {n: [] for n in RIG7}
anchors = []
for a, b in FRONT_PAIRS + REAR_PAIRS:
    pa, pb = match(a, b)
    if len(pa) < 2:
        continue
    ra, rb = rays_from_pixels(cams[RIG7.index(a)], pa), rays_from_pixels(cams[RIG7.index(b)], pb)
    T_rel = np.linalg.inv(T_c2r[RIG7.index(a)]) @ T_c2r[RIG7.index(b)]
    R, t = T_rel[:3, :3], T_rel[:3, 3]
    keep_a, za, keep_b, zb = [], [], [], []
    for i in range(len(ra)):
        lam, *_ = np.linalg.lstsq(np.stack([ra[i], -(R @ rb[i])], 1), t, rcond=None)
        if lam[0] <= 0.5 or lam[1] <= 0.5:
            continue
        p1, p2 = lam[0] * ra[i], t + lam[1] * (R @ rb[i])
        if np.linalg.norm(p1 - p2) > 0.5:
            continue
        X = (p1 + p2) / 2
        keep_a.append(pa[i]); za.append(X[2])
        keep_b.append(pb[i]); zb.append((np.linalg.inv(T_rel)[:3, :3] @ X
                                         + np.linalg.inv(T_rel)[:3, 3])[2])
    za, zb = np.array(za), np.array(zb)
    ga, gb = za < GATE_M, zb < GATE_M
    if ga.sum() >= MIN_PTS:
        ratios[a] += list(depth_at(depth[RIG7.index(a)], np.array(keep_a)[ga]) / za[ga])
        r_ = rays_from_pixels(cams[RIG7.index(a)], np.array(keep_a)[ga])
        pc = r_ / r_[:, 2:] * za[ga, None]
        anchors.append(pc @ T_c2r[RIG7.index(a)][:3, :3].T + T_c2r[RIG7.index(a)][:3, 3])
    if gb.sum() >= MIN_PTS:
        ratios[b] += list(depth_at(depth[RIG7.index(b)], np.array(keep_b)[gb]) / zb[gb])

s_cls = {n: float(np.median(v)) for n, v in ratios.items() if len(v) >= MIN_PTS}
for src, dst in [("front_wide", "front_tele"), ("rear_left_70fov", "rear_tele_30fov")]:
    if src in s_cls and dst not in s_cls:
        pw, pt = match(src, dst)
        dw = depth_at(depth[RIG7.index(src)], pw) / s_cls[src]
        dt = depth_at(depth[RIG7.index(dst)], pt)
        m = (dw > 0) & (dw < GATE_M) & (dt > 0)
        if m.sum() >= MIN_PTS:
            s_cls[dst] = float(np.median(dt[m] / dw[m]))
for n in RIG7:                                  # un-anchored -> front_wide's scale
    s_cls.setdefault(n, s_cls["front_wide"])
anchors = np.concatenate(anchors)
print("per-camera classical scale (divide depth by this):")
print({k[:10]: round(v, 3) for k, v in s_cls.items()})

## 6. Judge on LiDAR: metrics + BEV

In [ ]:
def sparse_gt(cam_i):
    Tinv = np.linalg.inv(T_c2r[cam_i])
    pc = pts_rig @ Tinv[:3, :3].T + Tinv[:3, 3]
    inc = np.degrees(np.arccos(np.clip(pc[:, 2] / np.maximum(
        np.linalg.norm(pc, axis=1), 1e-9), -1, 1)))
    m = (pc[:, 2] > 0.5) & (inc < MAX_INC7[RIG7[cam_i]])
    pc = pc[m]
    px = cams[cam_i].ray2pixel(pc / np.linalg.norm(pc, axis=1, keepdims=True))
    px = px[0] if isinstance(px, tuple) else np.asarray(px)
    u, v = px[:, 0] * Wp / NATIVE_W, px[:, 1] * Hp / NATIVE_H
    ok = (u >= 0) & (u < Wp) & (v >= 0) & (v < Hp)
    u, v, z = u[ok].astype(int), v[ok].astype(int), pc[ok, 2]
    gt = np.full((Hp, Wp), np.nan)
    o = np.argsort(-z)
    gt[v[o], u[o]] = z[o]                       # min-pool: nearest written last
    lm = minimum_filter(np.nan_to_num(gt, nan=np.inf), size=5)
    gt[np.isfinite(gt) & (gt > 1.15 * lm + 0.3)] = np.nan   # occlusion z-test
    return gt

print(f"{'camera':18s} {'AbsRel':>7s} {'d<1.25':>7s} {'bias':>6s} {'n':>7s}")
for i, n in enumerate(RIG7):
    gt = sparse_gt(i)
    m = np.isfinite(gt) & (gt > 1) & (gt < 80) & (depth[i] > 0)
    if m.sum() < 200:
        continue
    d, g = depth[i][m] / s_cls[n], gt[m]
    r = np.maximum(d / g, g / d)
    print(f"{n:18s} {np.mean(np.abs(d - g) / g):7.3f} {np.mean(r < 1.25):7.3f} "
          f"{np.median(d / g):6.2f} {int(m.sum()):7d}")

In [ ]:
def unproject_ftheta(depth_i, cam, T, conf_i, max_pts=50_000):
    v, u = np.meshgrid(np.arange(Hp), np.arange(Wp), indexing="ij")
    m = conf_i > np.median(conf_i)
    pix = np.stack([u[m] * NATIVE_W / Wp, v[m] * NATIVE_H / Hp], -1).astype(float)
    r = rays_from_pixels(cam, pix)
    pc = r / np.clip(r[:, 2:], 1e-3, None) * depth_i[m][:, None]
    pts = pc @ T[:3, :3].T + T[:3, 3]
    return pts[np.random.default_rng(0).permutation(len(pts))[:max_pts]]

def band(p):
    return ((p[:, 0] > -25) & (p[:, 0] < 30) & (np.abs(p[:, 1]) < 20)
            & (p[:, 2] > -0.5) & (p[:, 2] < 6))

lid = pts_rig[band(pts_rig)]
fig, ax = plt.subplots(figsize=(8, 9))
ax.scatter(-lid[:, 1], lid[:, 0], c="0.65", s=0.3, lw=0, alpha=0.5)
clouds = []
for i, n in enumerate(RIG7):
    pts = unproject_ftheta(depth[i] / s_cls[n], cams[i], T_c2r[i], conf[i])
    clouds.append(pts)
    m = band(pts)
    ax.scatter(-pts[m, 1], pts[m, 0], c=PAL7[i][None], s=0.25, lw=0, alpha=0.45,
               label=SHORT7[i])
am = (np.abs(anchors[:, 1]) < 20) & (anchors[:, 0] > -25) & (anchors[:, 0] < 30)
ax.scatter(-anchors[am, 1], anchors[am, 0], c="k", marker="*", s=14, lw=0,
           label=f"anchors (n={len(anchors)})")
P = np.concatenate(clouds); P = P[band(P)]
d, _ = cKDTree(lid).query(P[np.random.default_rng(0).permutation(len(P))[:100_000]], k=1)
ax.set_title(f"{PREFIX} — DA3 (+CAM), classical meter; LiDAR gray\n"
             f"median NN dist to LiDAR: {np.median(d[d < 5]):.2f} m")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(fontsize=7.5, loc="lower right")
ax.set_xlim(-20, 20); ax.set_ylim(-25, 30)
ax.set_xlabel("right (m)"); ax.set_ylabel("forward (m)")
plt.tight_layout(); plt.show()

## Notes

* Scale convention: `s_cls` = median(DA3 depth / triangulated z); **divide** depth by it.
* The camera-translation meter that `align_to_input_ext_scale=True` bakes in is
  2–5× short on this rig — that is what section 5 replaces. LiDAR here is a judge only.
* Full study (14 scenes, ablations, failure cases, honest limitations):
  [write-up](https://wbjang.github.io/blog/posts/av-rig-recon/).
* Data: [NVIDIA PhysicalAI-AV](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles)
  (gated; do not redistribute imagery). Model:
  [Depth Anything 3](https://github.com/ByteDance-Seed/Depth-Anything-3).